#**Geo-coder**
This is the geocoder module. The inputs to this module are the tweet texts and the output is a dataset containing the routes corresponding to each tweet text.

# **Steps to run this module**

1. It is assumed that you have downloaded the entire **replication** folder from the FigShare site, unzipped and extracted the files and uploaded the entire folder to your Google drive.

2. Run the cell below to mount your Google drive to the Colab runtime.

3. Then **set the path** to the location of the **replication** folder (including the replication folder) on your Google drive by assigning the appropriate folder location to the variable "REPL_PATH". For example "/content/drive/MyDrive/replication". (Note that you must include the replication folder in the path)

4. You may then run the code in the rest of the cells.



# **Mount Google Drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Set Path (once for each notebook)**

In [4]:
# Set path here
REPL_PATH = "/content/drive/MyDrive/.../replication"

#**Install Libraries**

In [5]:
!pip install spacy==3.7.2
!pip install osmnx
!pip install opencage
!pip install openrouteservice
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.15.1
    Uninstalling typer-0.15.1:
      Successfully uninstalled typer-0.15.1
  Attempting uninstall: smart-open
    Found existing installation: smart-open 7.1.0
    Uninstalling smart-open-7.1.0:
      Successfully uninstalled smart-open-7.1.0
  Attempting uninstall: cloudpathlib
    Found existing installation: cloudpathlib 0.20.0
    Uninstalling cloudpathlib-0.20.0:
      Successfully uninstalled cloudpathlib-0.20.0
  Attempting uninstall: weasel
    Found existing installation: weasel 0.4.1
    Uninstalling weasel-0.4.1:
     

In [6]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

## Custom Spacy Train

In [7]:
import spacy
from spacy.pipeline import EntityRuler

# Load the existing spaCy model
nlp = spacy.load("en_core_web_sm")

# Create the EntityRuler
ruler = EntityRuler(nlp, overwrite_ents=True)

# Read location terms from the file 'tweet_location_terms.txt'
patterns = []
with open(REPL_PATH+'/Data/tweet_location_terms.txt', 'r', encoding='utf-8') as f:
    for line in f:
        term = line.strip()
        if term:  # Ensure the line is not empty
            # Create a pattern for each term
            patterns.append({"label": "LOC", "pattern": term})

# Add patterns to the ruler
ruler.add_patterns(patterns)

# Add the EntityRuler to the pipeline before the 'ner' component
nlp.add_pipe('entity_ruler', before='ner')
nlp.get_pipe('entity_ruler').add_patterns(patterns)

# Save the EntityRuler patterns to disk (optional)
ruler.to_disk("ruler_patterns.json")

# Save the entire SpaCy model with the custom EntityRuler to disk
nlp.to_disk("saved_model")

##Non case-sensitive

In [8]:
import spacy
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span
import spacy.util

# Load the existing spaCy model
nlp = spacy.load("saved_model")

# Create the PhraseMatcher
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")  # Use LOWER to make it case-insensitive

# Read location terms and create Doc objects
with open(REPL_PATH+'/Data/tweet_location_terms.txt', 'r', encoding='utf-8') as f:
    terms = [line.strip() for line in f if line.strip()]
patterns = [nlp.make_doc(text) for text in terms]

# Add patterns to the matcher under the label "LOC"
matcher.add("LOC", patterns)

# Function to process tweets and add entities
def nlp_tweet(tweet):
    doc = nlp(tweet)
    matches = matcher(doc)
    spans = []
    for match_id, start, end in matches:
        span = Span(doc, start, end, label=match_id)
        spans.append(span)
    # Combine existing entities with new spans
    existing_ents = list(doc.ents)
    all_ents = existing_ents + spans
    # Filter out overlapping spans
    filtered_ents = spacy.util.filter_spans(all_ents)
    # Set the filtered entities back to the doc
    doc.ents = filtered_ents
    return doc

In [9]:
# Load the saved model
nlp = spacy.load("saved_model")

tweet = "Traffic Alert Sansad Marg has been closed by Local Police due to procession. Inconvenience regretted."

# Process the tweet
doc = nlp_tweet(tweet)

# Extract entities
for ent in doc.ents:
    print(ent.text, ent.label_)


Traffic Alert PERSON
Sansad Marg LOC


## Classify tweets

In [10]:
import pandas as pd
import re

# Load CSV data
df = pd.read_csv(REPL_PATH+"/Data/tweets_data.csv")

# Define patterns with associated numbers
patterns = {
    "Origin to Destination at a Point (O2D@P)": (r"\bfrom\b.*\b(towards|to)\b.*\b(at|near|over|beside)\b", 9),
    "Origin to Multiple Destinations (O2MD)": (r"\bfrom\b.*\b(towards|to)\b.*\b(and|&)\b", 8),
    "Origin to Destination along/on a Line (O2DonL)": (r"\b(?:on|along)\b.*\bfrom\b.*\b(towards|to)\b|\bfrom\b.*\b(towards|to)\b.*\b(?:on|along)\b", 10),
    "Point on a Line (PonL)": (r"\b(at)\b.*\bon\b", 11),
    "Point and a Destination (P&D)": (r"\bat\b.*\btowards\b", 12),
    "Origin to Destination (O2D)": (r"\b(from|starting at)\b.*\b(towards|to)\b", 7),
    "Multiple Line Locations (ML)": (r"\b(underpass|overpass|lane|street|road|marg|flyover|bridge|highway|pathway|bypass|expressway|carriageway)\b.*\b(underpass|overpass|lane|street|road|marg|flyover|bridge|highway|pathway|bypass|expressway|carriageway)\b", 4),
    "Line Location(s) (L)": (r"\b(underpass|overpass|lane|street|road|marg|flyover|bridge|highway|pathway|bypass|expressway|carriageway)\b", 3),
    "Multiple Point Locations (MP)": (r"\b(at|near)\b.*\band\b", 2),
    "Point Location(s) (P)": (r"\b(at|in)\b", 1),
    "Origin(O)": (r"\b(from|starting at)\b", 5),
    "Destination(D)": (r"\b(towards|to)\b", 6)
}

# Function to classify tweets
def classify_tweet(tweet):
    for category, (pattern, number) in patterns.items():
        if re.search(pattern, tweet, re.IGNORECASE):
            # if(number == 1):
            #     print(tweet)
            return number  # Return the number associated with the pattern
    return 0  # Return 0 if no category matches

# Apply classification to each tweet
df['Category_Number'] = df['translated_text'].apply(classify_tweet)

# Count occurrences in each category
category_counts = df['Category_Number'].value_counts()

# Print the results
print(category_counts)

<ipython-input-10-f3a9e2e44db1>:5: DtypeWarning: Columns (6,7,37,38,39,40,41,42,43,44,45,46,47,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(REPL_PATH+"/Data/tweets_data.csv")


Category_Number
7     1559
9     1013
3      803
10     594
4      351
1      246
5      192
6      116
8      108
0       69
12      54
11      21
2       18
Name: count, dtype: int64


## Code for configurations

### 1. extract_point_route
E.g. Traffic Alert Traffic is now normal at Chirag Delhi.

In [11]:
def extract_point_route(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts a single-point route based on the P configuration.

    Parameters:
        toponyms (list): List of place names or toponyms to geocode.
        geocoder (Geocoder object): Instance of geocoder with API key.
        delhi_center (tuple): The central coordinates for Delhi to aid in geocoding.

    Returns:
        point_route (list): List containing a single [lng, lat] coordinate.
    """
    print("extract_point_route is being called")
    point_route = []

    # Geocode each toponym and return the first valid coordinate within Delhi bounds
    for place in point_locations:
        loc_info = geocoder.geocode(place, proximity=delhi_center)
        if loc_info:
            # Assume the first result is the most relevant for simplicity
            lat = loc_info[0]['geometry']['lat']
            lng = loc_info[0]['geometry']['lng']
            point_route = [[lng, lat]]
            break  # Only need one point, so we stop after the first valid result

    return point_route


### 2. extract_mp_routes
E.g. Traffic is heavy near Patiala House Court and Saket Court.

In [12]:
from itertools import combinations

def extract_mp_routes(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts multiple routes for the MP configuration by constructing routes
    between each pair of points, resulting in nC2 probable routes.

    Parameters:
        point_coords (list): List of [lat, lng] coordinates for each point.
        clnt (Client object): OpenRouteService client for routing API.

    Returns:
        probable_routes (list): List of routes, each route being a list of [lng, lat] coordinates.
    """
    print("extract_mp_routes is being called")
    probable_routes = []

    print("point_coords: ",point_coords)

    # Generate all possible pairs of points (nC2) and ensure point_coords has more than one point
    if len(point_coords) < 2:
        print("\033[41mNot enough points for MP route extraction\033[0m")
        return probable_routes

    # Generate all unique pairs of points (nC2)
    point_pairs = list(combinations(point_coords, 2))
    print("Point pairs:", point_pairs)

    # Extract continuous route for each pair of points
    for pair in point_pairs:
        point1, point2 = pair  # Unpack the pair into point1 and point2
        try:
            # # Print each pair for clarity
            # print("Point pair:", pair)
            # print("Point 1:", point1)
            # print("Point 2:", point2)

            # Convert points to [lng, lat] format for ORS
            point1_coords = point1[::-1]
            point2_coords = point2[::-1]
            print("Point 1 (for ORS):", point1_coords)
            print("Point 2 (for ORS):", point2_coords)

            route = clnt.directions([point1_coords, point2_coords], profile='driving-car', format='geojson')
            route_coordinates = route['features'][0]['geometry']['coordinates']

            # Append the continuous route for this pair to probable_routes
            probable_routes.extend(route_coordinates)

        except Exception as e:
            print(f"\033[41mError obtaining route for points {point1} and {point2}: {e}\033[0m")
            continue  # Skip any pairs that encounter errors

    return probable_routes

### 3. extract_line_route
E.g. Traffic on Kasturba Gandhi Marg is affected due to a candle march.

In [13]:
import spacy
import osmnx as ox
import networkx as nx
import geopandas as gpd
from geopy.distance import geodesic

# Load spaCy model (make sure it's downloaded and available)
nlp = spacy.load("saved_model")

def extract_line_road(line):
    # Define the area where your road is located (e.g., a bounding box or city name)
    place = "Delhi, India"

    # Download road network for the area
    G = ox.graph_from_place(place, network_type='drive')  # 'drive' for drivable roads

    # Extract edges (road segments) as a GeoDataFrame
    edges = ox.graph_to_gdfs(G, nodes=False)

    # Filter for the specific road by name
    road_name = line  # Replace with the actual road name
    road_segments = edges[edges['name'] == road_name]

    # Check if the road was found
    if road_segments.empty:
        print(f"Road '{road_name}' not found in the specified area.")
    else:
        # To get the entire road as a single line, combine the segments
        road_geometry = road_segments.unary_union  # Combines into a single MultiLineString

        # Extract coordinates from the combined geometry in (latitude, longitude) format
        coordinates = []
        if road_geometry.geom_type == 'MultiLineString':
            for line in road_geometry.geoms:  # Access each LineString in the MultiLineString
                coordinates.extend([(lat, lon) for lon, lat in line.coords])
        elif road_geometry.geom_type == 'LineString':
            coordinates = [(lat, lon) for lon, lat in road_geometry.coords]

        # Display the list of coordinates in (latitude, longitude) format
        # print("Line coordinates as a list in (latitude, longitude):")
        # print(coordinates)  # This will display the output as a Python list
        return coordinates

def extract_line_route(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts routes for the Line (L) configuration by performing spatial queries
    on linear geographic features or drawing routes between given points.

    Parameters:
        tweet (str): The tweet text for context (not used in this function).
        point_locations (list): List of location names or toponyms for linear features.
        point_coords (list): List of [lat, lng] coordinates for each point.
        geocoder (Geocoder object): Geocoder client for spatial queries.
        clnt (Client object): OpenRouteService client for routing API.
        delhi_center (tuple): Central point coordinates for proximity-based searches.

    Returns:
        line_routes (list): List of routes, each being the linear geometry of the identified or drawn structure.
    """
    line_route = []

    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }

    # Case 1: Single line handling
    if len(point_locations) == 1:
        line = point_locations[0]
        print("line:", line)
        line_route = extract_line_road(line)
        return line_route


    # Case 2: Multiple points, find route using ORS
    elif len(point_locations) > 1:
        print("Multiple points detected, finding routes for point pairs.")
        for i in range(len(point_coords) - 1):
            point1 = point_coords[i][::-1]  # Convert to [lng, lat] format
            point2 = point_coords[i + 1][::-1]  # Convert to [lng, lat] format

            try:
                # Request route between consecutive points
                route = clnt.directions([point1, point2], profile='driving-car', format='geojson')
                line_route = route['features'][0]['geometry']['coordinates']
                line_route.extend(route_coordinates)

            except Exception as e:
                print(f"\033[41mError obtaining route between points {point1} and {point2}: {e}\033[0m")

    return line_route

### 4. extract_multiple_lines_route
e.g. Avoid intersections at Outer Circle Connaught Place, Sansad Marg, Ashoka Road due to Tazia Procession.

In [14]:
from shapely.geometry import LineString, MultiLineString
from shapely.ops import linemerge

def extract_multiple_lines_route(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts routes for the Multiple Lines (ML) configuration by retrieving
    individual linear geometries for each toponym and combining them into a
    collective set of probable routes.

    Parameters:
        point_locations (list): List of location names or toponyms for the linear features.
        geocoder (Geocoder object): Geocoder client to search for geographic features.
        clnt (Client object): OpenRouteService client for routing API.

    Returns:
        merged_route (LineString or MultiLineString): Merged linear geometry representing
                                                      combined routes of all identified structures.
    """
    line_route = []

    # Calculate and accumulate routes between each consecutive pair of points
    for i in range(len(point_coords) - 1):
        line1 = point_coords[i][::-1]  # Convert to [lng, lat]
        line2 = point_coords[i + 1][::-1]  # Convert to [lng, lat]
        # print("Point 1 (for ORS):", line1)
        # print("Point 2 (for ORS):", line2)

        try:
            # Get the route segment between consecutive points
            route = clnt.directions([line1, line2], profile='driving-car', format='geojson')
            route_coordinates = route['features'][0]['geometry']['coordinates']

            if line_route:
                # Avoid duplication by skipping the first coordinate of each new segment
                line_route.extend(route_coordinates[1:])
            else:
                # Add the full route for the first segment
                line_route.extend(route_coordinates)

        except Exception as e:
            print(f"\033[41mError obtaining route for lines {line1} and {line2}: {e}\033[0m")
            continue  # Skip any pairs that encounter errors

    return line_route


### 5. extract_origin_routes
E.g. Traffic Alert Got from Patel Chowk P.O. Now traffic is normal.

In [15]:
def extract_origin_routes(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts routes for the Origin (O) configuration, where only the starting point is defined.
    Uses the bounding box around the origin to evaluate probable routes in all directions
    within the bounding area.

    Parameters:
        point_location (str): Location name or toponym representing the origin of traffic.
        geocoder (Geocoder object): Geocoder client to search for geographic features.
        clnt (Client object): OpenRouteService client for routing API.

    Returns:
        probable_routes (list): List of routes starting from the origin within the bounding box.
    """
    probable_routes = []
    point_location = point_locations[0]
    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }

    # Step 1: Geocode the origin to get the coordinates and bounding box
    # loc_info = geocoder.geocode(point_location, proximity=delhi_center)

    loc_info = geocoder.geocode(point_location, proximity=delhi_center)
    if loc_info:
        for result in loc_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']

            # Check if the result is within the specified bounds for Delhi
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                start_point = [lng, lat]

                # Retrieve the bounding box for the start point's geometry
                if 'bounds' in result:
                    bounding_box = result['bounds']

                    # Access southwest and northeast coordinates for the bounding box
                    southwest = bounding_box['southwest']
                    northeast = bounding_box['northeast']

                    # Print or use the bounding box as needed
                    print("Bounding box for start point geometry:", southwest, northeast)

                     # Check if the bounding box is fully within Delhi bounds
                    if (delhi_bounds['southwest']['lat'] <= southwest['lat'] <= delhi_bounds['northeast']['lat'] and
                        delhi_bounds['southwest']['lng'] <= southwest['lng'] <= delhi_bounds['northeast']['lng'] and
                        delhi_bounds['southwest']['lat'] <= northeast['lat'] <= delhi_bounds['northeast']['lat'] and
                        delhi_bounds['southwest']['lng'] <= northeast['lng'] <= delhi_bounds['northeast']['lng']):

                        print("Bounding box within Delhi bounds:", southwest, northeast)
                        # break  # Stop after finding the first valid bounding box within Delhi

                    else:
                        print("Bounding box found, but it extends outside Delhi bounds.")

                # Step 2: Generate multiple routes from this origin point within the bounding box
                start_point = [lng, lat]
                print("start_point: ",start_point)
                end_points = generate_end_points_within_bbox(southwest, northeast, num_points=4)  # Sample 4 end points
                print("end_points: ",end_points)

                for end_point in end_points:
                    try:
                        route = clnt.directions([start_point, end_point], profile='driving-car', format='geojson')
                        route_coordinates = route['features'][0]['geometry']['coordinates']
                        probable_routes.extend(route_coordinates)
                    except ors_exceptions.ApiError as e:
                        print(f"API Error when routing from {start_point} to {end_point}: {e}")
                    except Exception as e:
                        print(f"Error processing route from {start_point} to {end_point}: {e}")
                break  # Stop after finding the first valid origin point

    return probable_routes

def generate_end_points_within_bbox(southwest, northeast, num_points=4):
    """
    Generates a list of random end points within a bounding box defined by southwest and northeast coordinates.

    Parameters:
        southwest (dict): Coordinates of the southwest corner of the bounding box.
        northeast (dict): Coordinates of the northeast corner of the bounding box.
        num_points (int): Number of end points to generate.

    Returns:
        end_points (list): List of coordinates within the bounding box.
    """
    import random
    end_points = []

    for _ in range(num_points):
        lat = random.uniform(southwest['lat'], northeast['lat'])
        lng = random.uniform(southwest['lng'], northeast['lng'])
        end_points.append([lng, lat])

    return end_points


### 6. extract_destination_routes
E.g. Traffic has been diverted towards Vasant Vihar due to water stress.

In [16]:
def extract_destination_routes(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts routes for the Origin (O) configuration, where only the starting point is defined.
    Uses the bounding box around the origin to evaluate probable routes in all directions
    within the bounding area.

    Parameters:
        point_location (str): Location name or toponym representing the origin of traffic.
        geocoder (Geocoder object): Geocoder client to search for geographic features.
        clnt (Client object): OpenRouteService client for routing API.

    Returns:
        probable_routes (list): List of routes starting from the origin within the bounding box.
    """
    probable_routes = []
    point_location = point_locations[0]
    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }

    # Step 1: Geocode the origin to get the coordinates and bounding box
    # loc_info = geocoder.geocode(point_location, proximity=delhi_center)

    loc_info = geocoder.geocode(point_location, proximity=delhi_center)
    if loc_info:
        for result in loc_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']

            # Check if the result is within the specified bounds for Delhi
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                end_point = [lng, lat]

                # Retrieve the bounding box for the end point's geometry
                if 'bounds' in result:
                    bounding_box = result['bounds']

                    # Access southwest and northeast coordinates for the bounding box
                    southwest = bounding_box['southwest']
                    northeast = bounding_box['northeast']

                    # Print or use the bounding box as needed
                    print("Bounding box for end point geometry:", southwest, northeast)

                     # Check if the bounding box is fully within Delhi bounds
                    if (delhi_bounds['southwest']['lat'] <= southwest['lat'] <= delhi_bounds['northeast']['lat'] and
                        delhi_bounds['southwest']['lng'] <= southwest['lng'] <= delhi_bounds['northeast']['lng'] and
                        delhi_bounds['southwest']['lat'] <= northeast['lat'] <= delhi_bounds['northeast']['lat'] and
                        delhi_bounds['southwest']['lng'] <= northeast['lng'] <= delhi_bounds['northeast']['lng']):

                        print("Bounding box within Delhi bounds:", southwest, northeast)
                        # break  # Stop after finding the first valid bounding box within Delhi

                    else:
                        print("Bounding box found, but it extends outside Delhi bounds.")

                # Step 2: Generate multiple routes from this origin point within the bounding box
                end_point = [lng, lat]
                print("end_point: ",end_point)

                # start_points = []
                # southwest = (southwest['lng'], southwest['lat'])
                # northeast = (northeast['lng'], northeast['lat'])
                # start_points.append(southwest)
                # start_points.append(northeast)


                start_points = generate_end_points_within_bbox(southwest, northeast, num_points=4)  # Sample 4 start points
                print("start_points: ",start_points)

                for start_point in start_points:
                    try:
                        route = clnt.directions([start_point, end_point], profile='driving-car', format='geojson')
                        route_coordinates = route['features'][0]['geometry']['coordinates']
                        probable_routes.extend(route_coordinates)
                    except ors_exceptions.ApiError as e:
                        print(f"API Error when routing from {start_point} to {end_point}: {e}")
                    except Exception as e:
                        print(f"Error processing route from {start_point} to {end_point}: {e}")
                break  # Stop after finding the first valid origin point

    return probable_routes

def generate_end_points_within_bbox(southwest, northeast, num_points=4):
    """
    Generates a list of random end points within a bounding box defined by southwest and northeast coordinates.

    Parameters:
        southwest (dict): Coordinates of the southwest corner of the bounding box.
        northeast (dict): Coordinates of the northeast corner of the bounding box.
        num_points (int): Number of end points to generate.

    Returns:
        end_points (list): List of coordinates within the bounding box.
    """
    import random
    start_points = []

    for _ in range(num_points):
        lat = random.uniform(southwest['lat'], northeast['lat'])
        lng = random.uniform(southwest['lng'], northeast['lng'])
        start_points.append([lng, lat])

    return start_points


### 7. extract_origin_to_destination_route
E.g. Traffic is heavy from Inderlok to Shastri Nagar.

In [25]:
import spacy

# Load spaCy model (make sure it's downloaded and available)
nlp = spacy.load("saved_model")

def extract_origin_destination(tweet):
    """
    Extracts the origin and destination locations from a tweet using spaCy,
    based on the presence of keywords like 'from', 'starting at', 'to', and 'towards'.

    Parameters:
        tweet (str): The text of the tweet.

    Returns:
        origin (str): Extracted origin location.
        destination (str): Extracted destination location.
    """
    doc = nlp_tweet(tweet)
    locations = []

    # Step 1: Extract all named entities that could be locations
    for ent in doc.ents:
        if ent.label_ in ["LOC", "GPE", "ORG", "FAC"]:
            locations.append(ent.text)

    # Step 2: Initialize origin and destination as None
    origin = None
    destination = None

    # # Step 3: Search for keywords and assign locations based on order of keywords
    # for i, token in enumerate(doc):
    #     if token.text.lower() in ["from", "starting at"] and i + 1 < len(doc):
    #         # Find the next location after 'from' or 'starting at'
    #         for loc in locations:
    #             if loc in tweet[token.idx:]:
    #                 origin = loc
    #                 if isinstance(origin, list):  # check if origin is a list
    #                     origin = origin[0]
    #                 break

    #     elif token.text.lower() in ["to", "towards"] and i + 1 < len(doc):
    #         # Find the next location after 'to' or 'towards'
    #         for loc in locations:
    #             if loc in tweet[token.idx:]:
    #                 destination = loc
    #                 if isinstance(destination, list):  # check if destination is a list
    #                     destination = destination[0]
    #                 break

    #     # Stop if both origin and destination are found
    #     if origin and destination:
    #         break

    # Iterate over tokens and assign origin and destination based on proximity and keywords
    for token in doc:
        if token.lower_ in ["from", "starting at"]:
            next_locs = [loc for loc in locations if loc in tweet[token.idx:]]  # Find all locations mentioned after the keyword
            if next_locs:
                origin = next_locs[0]  # Take the first mentioned location after the keyword as origin

        elif token.lower_ in ["to", "towards"]:
            next_locs = [loc for loc in locations if loc in tweet[token.idx:]]  # Find all locations mentioned after the keyword
            if next_locs:
                destination = next_locs[0]  # Take the first mentioned location after the keyword as destination

    print("Locations:", locations)  # For debugging

    # Assuming origin and destination might be lists and you want the first element
    print("*** THIS IS IT ***")

    if isinstance(origin, list):
        origin = origin[0] if origin else None  # Take the first element or None if the list is empty

    if isinstance(destination, list):
        destination = destination[0] if destination else None  # Take the first element or None if the list is empty

    print(destination)

    return origin, destination

def extract_origin_to_destination_route(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts the route for the Origin to Destination (O2D) configuration, where both origin and destination are specified.
    Executes a route query to determine the most probable route between the origin and destination points.

    Parameters:
        origin (str): Name or toponym of the origin location.
        destination (str): Name or toponym of the destination location.
        geocoder (Geocoder object): Geocoder client to search for geographic coordinates.
        clnt (Client object): OpenRouteService client for routing API.

    Returns:
        route_coordinates (list): List of coordinates representing the route between origin and destination.
    """

    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }

    origin, destination = extract_origin_destination(tweet)
    print("Origin:", origin)
    print("Destination:", destination)

    if not origin or not destination:
        return []

    # Step 1: Geocode origin
    origin_coords = None
    origin_info = geocoder.geocode(origin, proximity=delhi_center)
    if origin_info:
        for result in origin_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                origin_coords = [lng, lat]
                break  # Use the first valid location within bounds

    # Step 2: Geocode destination
    destination_coords = None
    destination_info = geocoder.geocode(destination, proximity=delhi_center)
    if destination_info:
        for result in destination_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                destination_coords = [lng, lat]
                break  # Use the first valid location within bounds

    # Step 3: Execute route query if both coordinates are available
    route_coordinates = []
    if origin_coords and destination_coords:
        try:
            route = clnt.directions([origin_coords, destination_coords], profile='driving-car', format='geojson')
            route_coordinates = route['features'][0]['geometry']['coordinates']
        except ors_exceptions.ApiError as e:
            print(f"API Error while routing from {origin} to {destination}: {e}")
        except Exception as e:
            print(f"Error processing route from {origin} to {destination}: {e}")

    return route_coordinates


### 8. extract_origin_to_multiple_destinations_route
E.g. Traffic Alert Traffic is heavy from Inderlok to Zakhira and Kanhaiya Nagar due to weekly market.

In [18]:
import spacy

# Load spaCy model (make sure it's downloaded and available)
nlp = spacy.load("saved_model")

def extract_origin_destination(tweet):
    """
    Extracts the origin and destination locations from a tweet using spaCy,
    based on the presence of keywords like 'from', 'starting at', 'to', and 'towards'.

    Parameters:
        tweet (str): The text of the tweet.

    Returns:
        origin (str): Extracted origin location.
        destinations (list): Extracted destination locations.
    """
    doc = nlp_tweet(tweet)
    locations = []

    # Step 1: Extract all named entities that could be locations
    for ent in doc.ents:
        if ent.label_ in ["LOC", "GPE", "ORG", "FAC"]:
            locations.append(ent.text)

    # Step 2: Initialize origin and destinations
    origin = None
    destinations = []

    # Step 3: Search for keywords and assign locations
    counter = 0;
    for i, token in enumerate(doc):
        # Check for origin location
        if token.text.lower() in ["from", "starting at"] and i + 1 < len(doc):
            # for org in locations:
            #     if org in tweet[token.idx:]:
            origin = locations[counter]
            if isinstance(origin, list):  # check if origin is a list
                        origin = origin[0]
            counter+=1

        # Check for destination locations
        elif token.text.lower() in ["to", "towards"]:
            # The next token after 'to' or 'towards' should be a destination
            # for des in locations:
            #     if des in tweet[token.idx:] and des != origin:  # Ensure it's not the origin
            if counter<len(locations) and locations[counter] not in destinations and locations[counter]!= origin:
                destinations.append(locations[counter])
                counter+=1

        elif token.text.lower() in ["and","&"]:
            if counter<len(locations) and locations[counter] not in destinations and locations[counter]!= origin:
                destinations.append(locations[counter])
            counter+=1
    print("Locations:", locations)  # For debugging
    return origin, destinations


def extract_origin_to_multiple_destinations_route(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts routes for the Origin to Multiple Destinations (O2MD) configuration, where traffic flows from a single origin to multiple destinations.
    Calculates routes from the origin to each destination and combines them into a set.

    Parameters:
        tweet (str): The text of the tweet.
        point_locations (list): Not used in this version but can be kept for further use.
        point_coords (list): Not used in this version but can be kept for further use.
        geocoder (Geocoder object): Geocoder client to search for geographic coordinates.
        clnt (Client object): OpenRouteService client for routing API.

    Returns:
        all_routes_coordinates (list): Combined list of coordinates for routes from the origin to each destination.
    """

    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }

    origin, destinations = extract_origin_destination(tweet)
    print("Origin:", origin)
    print("Destinations:", destinations)

    if origin is None or destinations is None:
        return []

    # Step 1: Geocode origin
    origin_coords = None
    origin_info = geocoder.geocode(origin, proximity=delhi_center)
    if origin_info:
        for result in origin_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                origin_coords = [lng, lat]
                break  # Use the first valid location within bounds

    # Initialize list to store all route coordinates
    all_routes_coordinates = []

    # Step 2: Geocode each destination and obtain routes from origin to each destination
    for destination in destinations:
        destination_coords = None
        destination_info = geocoder.geocode(destination, proximity=delhi_center)
        if destination_info:
            for result in destination_info:
                lat = result['geometry']['lat']
                lng = result['geometry']['lng']
                if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                    delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                    destination_coords = [lng, lat]
                    break  # Use the first valid location within bounds

        # Step 3: Calculate route if both origin and destination coordinates are valid
        if origin_coords and destination_coords:
            try:
                route = clnt.directions([origin_coords, destination_coords], profile='driving-car', format='geojson')
                route_coordinates = route['features'][0]['geometry']['coordinates']
                all_routes_coordinates.extend(route_coordinates)  # Add route to the combined set
            except ors_exceptions.ApiError as e:
                print(f"API Error while routing from {origin} to {destination}: {e}")
            except Exception as e:
                print(f"Error processing route from {origin} to {destination}: {e}")

    return all_routes_coordinates


### 9. extract_origin_to_destination_at_point_route
E.g. Obstruction in traffic from Safdurjung Enclave towards IIT Flyover due to breakdown of a truck at Green Park.

In [19]:
import spacy

# Load spaCy model (make sure it's downloaded and available)
nlp = spacy.load("saved_model")

def extract_origin_destination_landmark(tweet):
    """
    Extracts the origin, destination, and landmark locations from a tweet using spaCy,
    based on keywords like 'from', 'starting at', 'to', 'towards', 'at', 'near', etc.

    Parameters:
        tweet (str): The text of the tweet.

    Returns:
        origin (str): Extracted origin location.
        destination (str): Extracted destination location.
        landmark (str): Extracted landmark location.
    """
    doc = nlp_tweet(tweet)
    locations = []

    # Step 1: Extract all named entities that could be locations
    for ent in doc.ents:
        if ent.label_ in ["LOC", "GPE", "ORG", "FAC"]:
            locations.append(ent.text)

    # Step 2: Initialize origin, destination, and landmark as None
    origin = None
    destination = None
    landmark = None

    # Step 3: Search for keywords and assign locations based on order of keywords
    for i, token in enumerate(doc):
        if not origin and token.text.lower() in ["from", "starting at"] and i + 1 < len(doc):
            # Find the next location after 'from' or 'starting at'
            for loc in locations:
                if loc in tweet[token.idx:]:
                    origin = loc
                    if isinstance(origin, list):  # check if origin is a list
                        origin = origin[0]
                    break

        elif not destination and token.text.lower() in ["to", "towards"] and i + 1 < len(doc):
            # Find the next location after 'to' or 'towards'
            for loc in locations:
                if loc in tweet[token.idx:] and loc != origin:
                    destination = loc
                    if isinstance(destination, list):  # check if destination is a list
                        destination = destination[0]
                    break

        elif not landmark and token.text.lower() in ["at", "near", "over", "beside"] and i + 1 < len(doc):
            # Find the next location after 'at', 'near', etc.
            for loc in locations:
                if loc in tweet[token.idx:] and loc != origin and loc != destination:
                    landmark = loc
                    if isinstance(landmark, list):  # check if landmark is a list
                        landmark = landmark[0]
                    break

        # Stop if origin, destination, and landmark are found
        if origin and destination and landmark:
            break

    print("Locations:", locations)  # For debugging
    return origin, destination, landmark

def extract_origin_to_destination_at_point_route(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts routes for the Origin to Destination at a Point (O2D@P) configuration, incorporating a landmark as a waypoint.

    Parameters:
        origin (str): Name or toponym of the origin location.
        destination (str): Name or toponym of the destination location.
        landmark (str): Name or toponym of the landmark location.
        geocoder (Geocoder object): Geocoder client to search for geographic coordinates.
        clnt (Client object): OpenRouteService client for routing API.

    Returns:
        route_coordinates (list): List of coordinates for the route from origin to destination via the landmark.
    """
    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }

    origin, destination, landmark = extract_origin_destination_landmark(tweet)
    print("Origin:", origin)
    print("Destination:", destination)
    print("landmark:", landmark)

    if not origin or not destination:
        return []

    # Step 1: Geocode origin
    origin_coords = None
    origin_info = geocoder.geocode(origin, proximity=delhi_center)
    if origin_info:
        for result in origin_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                origin_coords = [lng, lat]
                break  # Use the first valid location within bounds

    # Step 2: Geocode destination
    destination_coords = None
    destination_info = geocoder.geocode(destination, proximity=delhi_center)
    if destination_info:
        for result in destination_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                destination_coords = [lng, lat]
                break  # Use the first valid location within bounds


    if not landmark:
        if origin_coords and destination_coords:
            try:
                route = clnt.directions([origin_coords, destination_coords], profile='driving-car', format='geojson')
                route_coordinates = route['features'][0]['geometry']['coordinates']
            except ors_exceptions.ApiError as e:
                print(f"API Error while routing from {origin} to {destination}: {e}")
            except Exception as e:
                print(f"Error processing route from {origin} to {destination}: {e}")

            return route_coordinates
        else:
            return []

    # Step 3: Geocode landmark
    landmark_coords = None
    landmark_info = geocoder.geocode(landmark, proximity=delhi_center)
    if landmark_info:
        for result in landmark_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                landmark_coords = [lng, lat]
                break  # Use the first valid location within bounds

    # Step 4: Calculate route if all coordinates are valid
    route_coordinates = []
    if origin_coords and destination_coords and landmark_coords:
        try:
            # Create route with the landmark as a waypoint
            route = clnt.directions(
                [origin_coords, landmark_coords, destination_coords],
                profile='driving-car',
                format='geojson'
            )
            route_coordinates = route['features'][0]['geometry']['coordinates']
            return route_coordinates
        except ors_exceptions.ApiError as e:
            print(f"API Error while routing from {origin} to {destination} via {landmark}: {e}")
        except Exception as e:
            print(f"Error processing route from {origin} to {destination} via {landmark}: {e}")

    if origin_coords and destination_coords:
        try:
            # Create route with the landmark as a waypoint
            route = clnt.directions(
                [origin_coords, destination_coords],
                profile='driving-car',
                format='geojson'
            )
            route_coordinates = route['features'][0]['geometry']['coordinates']
            return route_coordinates
        except ors_exceptions.ApiError as e:
            print(f"API Error while routing from {origin} to {destination} via {landmark}: {e}")
        except Exception as e:
            print(f"Error processing route from {origin} to {destination} via {landmark}: {e}")
    return route_coordinates

### 10. extract_origin_to_destination_on_line_route
E.g. Traffic movement on Sansad Marg in the carriageway from Patel Chowk towards Jantar Mantar is closed due to demonstration.

In [20]:
import spacy
import osmnx as ox
import networkx as nx
import geopandas as gpd
from geopy.distance import geodesic

# Load spaCy model (make sure it's downloaded and available)
nlp = spacy.load("saved_model")

def extract_line_road(line):
    # Define the area where your road is located (e.g., a bounding box or city name)
    place = "Delhi, India"

    # Download road network for the area
    G = ox.graph_from_place(place, network_type='drive')  # 'drive' for drivable roads

    # Extract edges (road segments) as a GeoDataFrame
    edges = ox.graph_to_gdfs(G, nodes=False)

    # Filter for the specific road by name
    road_name = line  # Replace with the actual road name
    road_segments = edges[edges['name'] == road_name]

    # Check if the road was found
    if road_segments.empty:
        print(f"Road '{road_name}' not found in the specified area.")
    else:
        # To get the entire road as a single line, combine the segments
        road_geometry = road_segments.unary_union  # Combines into a single MultiLineString

        # Extract coordinates from the combined geometry in (latitude, longitude) format
        coordinates = []
        if road_geometry.geom_type == 'MultiLineString':
            for line in road_geometry.geoms:  # Access each LineString in the MultiLineString
                coordinates.extend([(lat, lon) for lon, lat in line.coords])
        elif road_geometry.geom_type == 'LineString':
            coordinates = [(lat, lon) for lon, lat in road_geometry.coords]

        # Display the list of coordinates in (latitude, longitude) format
        # print("Line coordinates as a list in (latitude, longitude):")
        # print(coordinates)  # This will display the output as a Python list
        return coordinates


def extract_origin_destination_line(tweet):
    """
    Extracts the origin, destination, and line locations from a tweet using spaCy,
    based on keywords like 'from', 'starting at', 'to', 'towards', 'on', etc.

    Parameters:
        tweet (str): The text of the tweet.

    Returns:
        origin (str): Extracted origin location.
        destination (str): Extracted destination location.
        line (str): Extracted line location.
    """
    doc = nlp_tweet(tweet)
    locations = []

    # Step 1: Extract all named entities that could be locations
    for ent in doc.ents:
        if ent.label_ in ["LOC", "GPE", "ORG", "FAC"]:
            locations.append(ent.text)

    # Step 2: Initialize origin, destination, and line as None
    origin = None
    destination = None
    line = None

    # Step 3: Search for keywords and assign locations based on order of keywords
    for i, token in enumerate(doc):
        if not origin and token.text.lower() in ["from", "starting at"] and i + 1 < len(doc):
            # Find the next location after 'from' or 'starting at'
            for loc in locations:
                if loc in tweet[token.idx:]:
                    origin = loc
                    if isinstance(origin, list):  # check if origin is a list
                        origin = origin[0]
                    break

        elif not destination and token.text.lower() in ["to", "towards"] and i + 1 < len(doc):
            # Find the next location after 'to' or 'towards'
            for loc in locations:
                if loc in tweet[token.idx:] and loc != origin:
                    destination = loc
                    if isinstance(destination, list):  # check if destination is a list
                        destination = destination[0]
                    break

        elif not line and token.text.lower() in ["on"] and i + 1 < len(doc):
            # Find the next location after 'at', 'near', etc.
            for loc in locations:
                if loc in tweet[token.idx:] and loc != origin and loc != destination:
                    line = loc
                    if isinstance(line, list):  # check if destination is a list
                        line = line[0]
                    break

        # Stop if origin, destination, and line are found
        if origin and destination and line:
            break

    print("Locations:", locations)  # For debugging
    return origin, destination, line

def extract_origin_to_destination_on_line_route(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts routes for the Origin to Destination along a Line (O2DonL) configuration.

    Parameters:
        origin (str): Name or toponym of the origin location.
        destination (str): Name or toponym of the destination location.
        line (str): Name of the road or pathway along which the traffic flow is specified.
        geocoder (Geocoder object): Geocoder client to search for geographic coordinates.
        clnt (Client object): OpenRouteService client for routing API.

    Returns:
        route_coordinates (list): List of coordinates for the route along the specified road.
    """

    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }
    line_route = []

    origin, destination, line = extract_origin_destination_line(tweet)
    print("Origin:", origin)
    print("Destination:", destination)
    print("line:", line)

    if not origin or not destination:
        return []


    # Step 1: Geocode origin
    origin_coords = None
    origin_info = geocoder.geocode(origin, proximity=delhi_center)
    if origin_info:
        for result in origin_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                origin_coords = [lat, lng]
                break  # Use the first valid location within bounds

    # Step 2: Geocode destination
    destination_coords = None
    destination_info = geocoder.geocode(destination, proximity=delhi_center)
    if destination_info:
        for result in destination_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                destination_coords = [lat, lng]
                break  # Use the first valid location within bounds

    # Step 3: Geocode line_route
    if line:
        line_route = extract_line_road(line)

    # Prepare for fallback and ensure no uninitialized variable usage
    new_origin_coords = None
    new_destination_coords = None

    # Step 5: Calculate origin and destination coords on line_route

    if line_route:
        new_origin_coords = find_closest_coordinate(origin_coords, line_route)
        new_destination_coords = find_closest_coordinate(destination_coords, line_route)


    # Step 5: Execute route query if both coordinates are available
    route_coordinates = []

    print(new_origin_coords, new_destination_coords)

    # Get route coordinates and print them
    if new_origin_coords and new_destination_coords:
        route_coordinates = get_route_coordinates(new_origin_coords, new_destination_coords)

    if not route_coordinates:
        if new_origin_coords and new_destination_coords:
            try:
                route = clnt.directions([origin_coords, destination_coords], profile='driving-car', format='geojson')
                route_coordinates = route['features'][0]['geometry']['coordinates']
                return route_coordinates
            except ors_exceptions.ApiError as e:
                print(f"API Error while routing from {origin} to {destination}: {e}")
            except Exception as e:
                print(f"Error processing route from {origin} to {destination}: {e}")

        if not route_coordinates:
            if origin_coords and destination_coords:
                try:
                    route = clnt.directions([origin_coords, destination_coords], profile='driving-car', format='geojson')
                    route_coordinates = route['features'][0]['geometry']['coordinates']
                except ors_exceptions.ApiError as e:
                    print(f"API Error while routing from {origin} to {destination}: {e}")
                except Exception as e:
                    print(f"Error processing route from {origin} to {destination}: {e}")

    return route_coordinates

def find_closest_coordinate(input_coord, line_route):
    """
    Finds the closest coordinate from a list of coordinates (line_route) to the input coordinate.

    Parameters:
        input_coord (tuple): The origin or destination coordinate as (latitude, longitude).
        line_route (list): List of coordinates (latitude, longitude) representing the line route.

    Returns:
        tuple: The closest coordinate from line_route to the input_coord.
    """
    # Initialize variables to track the closest coordinate and minimum distance
    closest_coord = None
    min_distance = float('inf')

    for coord in line_route:
        # Calculate the geodesic distance between input_coord and each coord in line_route
        distance = geodesic(input_coord, coord).meters

        # Update the closest coordinate if the current distance is less than the previous minimum distance
        if distance < min_distance:
            min_distance = distance
            closest_coord = coord

    return closest_coord

def get_route_coordinates(origin, destination):
    """
    Finds the route coordinates between two points using the road network graph.

    Parameters:
        origin (tuple): Latitude and longitude of the starting point.
        destination (tuple): Latitude and longitude of the ending point.

    Returns:
        list: Sequence of (latitude, longitude) coordinates along the route.
    """
    try:
        # Load the road network graph
        G = ox.graph_from_place("Delhi, India", network_type='drive')

        # Find the nearest nodes on the graph to the origin and destination
        origin_node = ox.nearest_nodes(G, origin[1], origin[0])
        destination_node = ox.nearest_nodes(G, destination[1], destination[0])

        # Calculate the shortest path between the two nodes
        route = nx.shortest_path(G, origin_node, destination_node, weight='length')

        # Extract the coordinates of the nodes in the route
        route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in route]

        return route_coords

    except nx.NetworkXNoPath:
        print(f"No path between {origin_node} and {destination_node}.")
        return []

### 11. extract_point_on_line_route
E.g. Water logging at Adchini on Aurobindo Marg. Kindly avoid this stretch. Inconvenience is regretted.”

In [21]:
import spacy
import osmnx as ox
import networkx as nx
import geopandas as gpd
from geopy.distance import geodesic
import math

# Load spaCy model (make sure it's downloaded and available)
nlp = spacy.load("saved_model")

def extract_line_road(line):
    # Define the area where your road is located (e.g., a bounding box or city name)
    place = "Delhi, India"

    # Download road network for the area
    G = ox.graph_from_place(place, network_type='drive')  # 'drive' for drivable roads

    # Extract edges (road segments) as a GeoDataFrame
    edges = ox.graph_to_gdfs(G, nodes=False)

    # Filter for the specific road by name
    road_name = line  # Replace with the actual road name
    road_segments = edges[edges['name'] == road_name]

    # Check if the road was found
    if road_segments.empty:
        print(f"Road '{road_name}' not found in the specified area.")
    else:
        # To get the entire road as a single line, combine the segments
        road_geometry = road_segments.unary_union  # Combines into a single MultiLineString

        # Extract coordinates from the combined geometry in (latitude, longitude) format
        coordinates = []
        if road_geometry.geom_type == 'MultiLineString':
            for line in road_geometry.geoms:  # Access each LineString in the MultiLineString
                coordinates.extend([(lat, lon) for lon, lat in line.coords])
        elif road_geometry.geom_type == 'LineString':
            coordinates = [(lat, lon) for lon, lat in road_geometry.coords]

        # Display the list of coordinates in (latitude, longitude) format
        # print("Line coordinates as a list in (latitude, longitude):")
        # print(coordinates)  # This will display the output as a Python list
        return coordinates


def extract_point_line(tweet):
    """
    Extracts the point and line locations from a tweet using spaCy,
    based on keywords like 'at', 'on', etc.

    Parameters:
        tweet (str): The text of the tweet.

    Returns:
        point (str): Extracted point location.
        line (str): Extracted line location.
    """
    doc = nlp_tweet(tweet)
    locations = []

    # Step 1: Extract all named entities that could be locations
    for ent in doc.ents:
        if ent.label_ in ["LOC", "GPE", "ORG", "FAC"]:
            locations.append(ent.text)

    # Step 2: Initialize point and line as None
    point = None
    line = None

    # Step 3: Search for keywords and assign locations based on order of keywords
    for i, token in enumerate(doc):
        if not point and token.text.lower() in ["at"] and i + 1 < len(doc):
            # Find the next location after 'from' or 'starting at'
            for loc in locations:
                if loc in tweet[token.idx:]:
                    point = loc
                    if isinstance(point, list):  # check if point is a list
                        point = point[0]
                    break

        elif not line and token.text.lower() in ["on"] and i + 1 < len(doc):
            # Find the next location after 'on', etc.
            for loc in locations:
                if loc in tweet[token.idx:] and loc != point:
                    line = loc
                    if isinstance(line, list):  # check if line is a list
                        line = line[0]
                    break

        # Stop if point, and line are found
        if point and line:
            break

    print("Locations:", locations)  # For debugging
    return point, line

def extract_point_on_line_route(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Extracts routes for the Point on a Line (PonL) configuration.

    Parameters:
        landmark (str): Name of the landmark or toponym.
        road_name (str): Name of the road or pathway near which the landmark is located.
        geocoder (Geocoder object): Geocoder client to search for geographic coordinates.
        clnt (Client object): OpenRouteService client for routing API.

    Returns:
        route_coordinates (list): List of coordinates for the route passing through or connected to the landmark.
    """

    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }

    point, line = extract_point_line(tweet)
    print("Point:", point)
    print("line:", line)

    if not point or not line:
        return []

    # Step 1: Geocode point
    point_coords = None
    origin_info = geocoder.geocode(point, proximity=delhi_center)
    if origin_info:
        for result in origin_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                point_coords = [lat, lng]
                break  # Use the first valid location within bounds


    line_route = []
    route_coordinates = []

    # Step 3: Geocode line_route
    if line:
        line_route = extract_line_road(line)
    print("point_coords: ", point_coords)

    if not line_route or not point_coords:
        return route_coordinates

    # Step 4: Calculate line route which passes through the point
    # origin, destination = find_farthest_points_in_opposite_directions(line_route, point_coords)
    # waypoint = point_coords

    # Step 4: Calculate line route which passes through the point
    farthest_points = find_farthest_points_in_opposite_directions(line_route, point_coords)

    if farthest_points is None:
        print("Failed to find two farthest points in opposite directions.")
        return []  # Return an empty list or appropriate default

    origin, destination = farthest_points
    waypoint = point_coords

    # Get route coordinates through the waypoint and print them
    route_coordinates = get_route_with_waypoint(origin, waypoint, destination)

    # Step 5: Execute route query if both coordinates are available
    if not route_coordinates:
        if origin and destination and waypoint:
            try:
                # Create route with the landmark as a waypoint
                route = clnt.directions(
                    [origin, waypoint, destination],
                    profile='driving-car',
                    format='geojson'
                )
                route_coordinates = route['features'][0]['geometry']['coordinates']
                return route_coordinates
            except ors_exceptions.ApiError as e:
                print(f"API Error while routing from {origin} to {destination} via {waypoint}: {e}")
            except Exception as e:
                print(f"Error processing route from {origin} to {destination} via {waypoint}: {e}")

        if origin and destination:
            try:
                # Create route with the landmark as a waypoint
                route = clnt.directions(
                    [origin, destination],
                    profile='driving-car',
                    format='geojson'
                )
                route_coordinates = route['features'][0]['geometry']['coordinates']
                return route_coordinates
            except ors_exceptions.ApiError as e:
                print(f"API Error while routing from {origin} to {destination} via {waypoint}: {e}")
            except Exception as e:
                print(f"Error processing route from {origin} to {destination} via {waypoint}: {e}")
    return route_coordinates

def calculate_bearing(point1, point2):
    """
    Calculates the initial bearing (angle) from point1 to point2.

    Parameters:
        point1 (tuple): The (latitude, longitude) of the first point.
        point2 (tuple): The (latitude, longitude) of the second point.

    Returns:
        float: Bearing in degrees.
    """
    lat1, lon1 = math.radians(point1[0]), math.radians(point1[1])
    lat2, lon2 = math.radians(point2[0]), math.radians(point2[1])

    d_lon = lon2 - lon1
    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - (math.sin(lat1) * math.cos(lat2) * math.cos(d_lon))
    bearing = math.degrees(math.atan2(x, y))
    return (bearing + 360) % 360  # Normalize to 0-360 degrees

def find_farthest_points_in_opposite_directions(line_route, point_coordinate):
    """
    Finds two points in opposite directions from the point_coordinate that are the farthest away on the line_route.

    Parameters:
        line_route (list of tuples): List of (latitude, longitude) coordinates defining the road.
        point_coordinate (tuple): The (latitude, longitude) of the point of interest.

    Returns:
        tuple: Two farthest coordinates in opposite directions from point_coordinate.
    """

    # Calculate distances of each point on line_route from point_coordinate
    distances = [(geodesic(point, point_coordinate).meters, point) for point in line_route]

    # Sort distances in descending order
    distances.sort(reverse=True, key=lambda x: x[0])

    # Initialize two farthest points in opposite directions
    farthest_point_1 = distances[0][1]
    farthest_point_2 = None

    for dist, point in distances:
        # Check if point is in the opposite direction of farthest_point_1
        # Calculate the bearing (angle) between the point_coordinate and potential farthest points
        bearing1 = calculate_bearing(point_coordinate, farthest_point_1)
        bearing2 = calculate_bearing(point_coordinate, point)
        angle_difference = abs(bearing1 - bearing2) % 360

        # Opposite direction condition: Angle difference should be close to 180 degrees
        if 100 <= angle_difference <= 260:
            farthest_point_2 = point
            break

    # Ensure we have found two points in opposite directions
    if farthest_point_2 is None:
        print("Couldn't find two farthest points in opposite directions.")
        return None

    return farthest_point_1, farthest_point_2

def get_route_with_waypoint(origin, waypoint, destination):
    """
    Finds the route coordinates between origin and destination via a waypoint using the road network graph.

    Parameters:
        origin (tuple): Latitude and longitude of the starting point.
        waypoint (tuple): Latitude and longitude of the waypoint.
        destination (tuple): Latitude and longitude of the ending point.

    Returns:
        list: Sequence of (latitude, longitude) coordinates along the entire route passing through the waypoint.
    """

    # # Validate that coordinates are floats
    # origin = (float(origin[0]), float(origin[1]))
    # waypoint = (float(waypoint[0]), float(waypoint[1]))
    # destination = (float(destination[0]), float(destination[1]))

    # Load the road network graph
    G = ox.graph_from_place("Delhi, India", network_type='drive')

    # Find the nearest nodes on the graph to the origin, waypoint, and destination
    origin_node = ox.nearest_nodes(G, origin[1], origin[0])
    waypoint_node = ox.nearest_nodes(G, waypoint[1], waypoint[0])
    destination_node = ox.nearest_nodes(G, destination[1], destination[0])

    # Calculate shortest paths for each segment
    route_origin_to_waypoint = nx.shortest_path(G, origin_node, waypoint_node, weight='length')
    route_waypoint_to_destination = nx.shortest_path(G, waypoint_node, destination_node, weight='length')

    # Combine the routes, excluding the duplicate waypoint node
    full_route = route_origin_to_waypoint + route_waypoint_to_destination[1:]

    # Extract the coordinates of the nodes in the combined route
    route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in full_route]

    return route_coords


### 12. extract_point_and_destination_route
E.g. Traffic Alert Traffic is now normal at Tilak Nagar towards Hari Nagar.

In [22]:
import spacy

# Load spaCy model (make sure it's downloaded and available)
nlp = spacy.load("saved_model")

def extract_point_direction(tweet):
    """
    Extracts the point location and direction from a tweet using spaCy,
    based on the presence of keywords like 'at', 'to', and 'towards'.

    Parameters:
        tweet (str): The text of the tweet.

    Returns:
        point (str): Extracted point.
        direction (str): Extracted direction.
    """
    doc = nlp_tweet(tweet)
    locations = []

    # Step 1: Extract all named entities that could be locations
    for ent in doc.ents:
        if ent.label_ in ["LOC", "GPE", "ORG", "FAC"]:
            locations.append(ent.text)

    # Step 2: Initialize point and direction as None
    point = None
    direction = None

    # Step 3: Search for keywords and assign locations based on order of keywords
    for i, token in enumerate(doc):
        if token.text.lower() in ["at"] and i + 1 < len(doc):
            # Find the next location after 'from' or 'starting at'
            for loc in locations:
                if loc in tweet[token.idx:]:
                    point = loc
                    if isinstance(point, list):  # check if point is a list
                        point = point[0]
                    break

        elif token.text.lower() in ["to", "towards"] and i + 1 < len(doc):
            # Find the next location after 'to' or 'towards'
            for loc in locations:
                if loc in tweet[token.idx:]:
                    direction = loc
                    if isinstance(direction, list):  # check if direction is a list
                        direction = direction[0]
                    break

        # Stop if both point and direction are found
        if point and direction:
            break

    print("Locations:", locations)  # For debugging
    return point, direction

def extract_point_and_destination_route(tweet, point_locations, point_coords, geocoder, clnt, delhi_center):
    """
    Finds the extended route based on the point and a directional preposition.

    Parameters:
        - point_coordinate: The coordinate of the point as [lat, lng].
        - line_route: List of coordinates representing the line route [[lat1, lng1], [lat2, lng2], ...].
        - direction: Direction keyword ('towards' or 'from') indicating the desired route extension.

    Returns:
        - directional_route: The portion of the route extending in the specified direction.
    """

    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }

    point, direction = extract_point_direction(tweet)
    print("point:", point)
    print("direction:", direction)

    if direction is None:
        return []


    # Step 1: Geocode point
    point_coords = None
    point_info = geocoder.geocode(point, proximity=delhi_center)
    if point_info:
        for result in point_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                point_coords = [lng, lat]
                break  # Use the first valid location within bounds

    # Step 2: Geocode direction
    direction_coords = None
    direction_info = geocoder.geocode(direction, proximity=delhi_center)
    if direction_info:
        for result in direction_info:
            lat = result['geometry']['lat']
            lng = result['geometry']['lng']
            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                direction_coords = [lng, lat]
                break  # Use the first valid location within bounds

    # Step 3: Execute route query if both coordinates are available
    route_coordinates = []
    if point_coords and direction_coords:
        try:
            route = clnt.directions([point_coords, direction_coords], profile='driving-car', format='geojson')
            route_coordinates = route['features'][0]['geometry']['coordinates']
        except ors_exceptions.ApiError as e:
            print(f"API Error while routing from {point} to {direction}: {e}")
        except Exception as e:
            print(f"Error processing route from {point} to {direction}: {e}")

    return route_coordinates


## The Geocoder

In [ ]:
import pandas as pd
import spacy
import nltk
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from spacy.matcher import Matcher
from opencage.geocoder import OpenCageGeocode
from openrouteservice import client, exceptions as ors_exceptions
import folium

# Load the entire SpaCy model from disk
nlp = spacy.load("saved_model")

# Load CSV to get fallback bounding boxes
df = pd.read_csv(REPL_PATH+"/Data/tweets_data.csv")

# Counter for skipped tweets
skipped_tweets = 0
tweet_counter = 0

def process_tweet(tweet, bbox_ne_lat, bbox_ne_lon, bbox_sw_lat, bbox_sw_lon):
    global skipped_tweets, tweet_counter
    route_coordinates = []
    print(f"Skipped {skipped_tweets} tweets till now.")

    # Print the tweet with yellow background and black text
    print(f"\033[43mProcessing Tweet {tweet_counter}: {tweet}\033[0m")
    tweet_counter += 1

    # Step 1: Tokenization
    doc = nlp_tweet(tweet)
    tokens = [token.text for token in doc if not token.is_punct and not token.is_space]
    print("Tokenized Words:", tokens)

    # Step 2: POS Tagging with NLTK
    nltk_tokens = word_tokenize(tweet)
    pos_tags = pos_tag(nltk_tokens)
    print("POS Tags (NLTK):", pos_tags)


    # Step 3: SpaCy Entity Extraction
    locations = []
    for ent in doc.ents:
        if ent.label_ in ["LOC", "GPE", "ORG", "FAC"]:
            locations.append(ent.text)

    for ent in doc.ents:
         print(ent.text, ent.label_)

    print("Locations:", locations)

    # Check if no locations were found
    if not locations:
        # Print the message with red background and white text
        print("\033[41mNo location found, cannot be geocoded\033[0m")

        skipped_tweets += 1
        return None, None  # Skip processing for this tweet

    # Continue with the rest of the function if locations were found

    # Extracted prepositions (not used in routing but useful for analysis)
    # Step 3.a: Extract Prepositions
    prepositions = [token.text for token in doc if token.dep_ == "prep"]
    print("Extracted Prepositions:", prepositions)

    # Step 3.b: Classify Prepositions
    direction_prepositions = ["from", "towards", "to"]
    place_prepositions = ["near", "above", "under", "besides"]
    for token in doc:
        if token.dep_ == "prep":
            if token.text.lower() in direction_prepositions:
                print("Directional Preposition:", token.text.lower())
            elif token.text.lower() in place_prepositions:
                print("Place Preposition:", token.text.lower())


    # Check if the word 'remove' is in the tweet
    is_removed = 'remove' in tweet.lower()

    # Classify locations as locations
    point_locations = []

    for location in locations:
        point_locations.append(location)


    # Step 4: OpenCage Geocoding
    api_key = "e458129e37ce462488f88e9fb747f6fa"    #faiz
    geocoder = OpenCageGeocode(api_key)
    delhi_center = (28.6139, 77.2090)  # Latitude and longitude of central Delhi

    # Define a bounding box around Delhi for stricter proximity checking
    delhi_bounds = {
        'southwest': {'lat': 28.4041, 'lng': 76.8377},
        'northeast': {'lat': 28.8832, 'lng': 77.6789}
    }

    # Geocode point locations
    point_coords = []
    for point in point_locations:
        loc_info = geocoder.geocode(point, proximity=delhi_center)
        if loc_info:
            for result in loc_info:
                lat = result['geometry']['lat']
                lng = result['geometry']['lng']
                # Check if the result is within the specified bounds for Delhi
                if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                    delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                    point_coords.append((lat, lng))
                    break  # Assume first valid result is acceptable

    # Step 5: Extract Routes using OpenrouteService
    api_key_ors = "5b3ce3597851110001cf6248d42d05dd7ddf403aa61993f105eb5f37"  # farazidris786
    clnt = client.Client(key=api_key_ors)

    # Create a dictionary to map configuration numbers to functions
    config_functions = {
        1: extract_point_route,
        2: extract_mp_routes,
        3: extract_line_route,
        4: extract_multiple_lines_route,
        5: extract_origin_routes,
        6: extract_destination_routes,
        7: extract_origin_to_destination_route,
        8: extract_origin_to_multiple_destinations_route,
        9: extract_origin_to_destination_at_point_route,
        10: extract_origin_to_destination_on_line_route,
        11: extract_point_on_line_route,
        12: extract_point_and_destination_route,
    }

    config_number = classify_tweet(tweet)
    print("config_num: ",config_number)

    # Dispatcher function to call the correct configuration function
    def handle_configuration(config_number):
        if config_number in config_functions:
            route_coordinates = config_functions[config_number](tweet, point_locations, point_coords, geocoder, clnt, delhi_center)
            return route_coordinates
        else:
            print(f"Configuration {config_number} not defined.")
            return None

    # Call the handle_configuration function to process based on config_number
    route_coordinates = handle_configuration(config_number)

    # Create a route that connects all point locations
    if not route_coordinates:
        print("IT'S ME WHO'S DOING IT NOW")
        try:
            if len(point_coords) > 1:
                coordinates = [coord[::-1] for coord in point_coords]  # Convert to [lng, lat] format for ORS
                route = clnt.directions(coordinates, profile='driving-car', format='geojson', continue_straight=False)
                route_coordinates = route['features'][0]['geometry']['coordinates']
            elif len(point_coords) == 1 or len(point_locations) == 1 or (len(point_coords) == 0 and len(point_locations) >= 1):
                # Geocode point locations and extract routes
                for point in point_locations:
                    loc_info = geocoder.geocode(point, proximity=delhi_center)
                    valid_location_found = False
                    if loc_info:
                        for result in loc_info:
                            lat = result['geometry']['lat']
                            lng = result['geometry']['lng']
                            # Check if the result is within the specified bounds for Delhi
                            if (delhi_bounds['southwest']['lat'] <= lat <= delhi_bounds['northeast']['lat'] and
                                delhi_bounds['southwest']['lng'] <= lng <= delhi_bounds['northeast']['lng']):
                                if 'bounds' in result:
                                    southwest = result['bounds']['southwest']
                                    northeast = result['bounds']['northeast']
                                    # Create a small route using the southwest and northeast corners of the bounding box
                                    bounding_box_route = [
                                        [southwest['lng'], southwest['lat']],
                                        [northeast['lng'], northeast['lat']]
                                    ]
                                    route = clnt.directions(bounding_box_route, profile='driving-car', format='geojson', continue_straight=False)
                                    route_coordinates = route['features'][0]['geometry']['coordinates']
                                    valid_location_found = True
                                    break  # Use the first valid bounding box for route creation
                        print(valid_location_found)
                    if not valid_location_found:
                        # Define fallback bounding box from CSV if no valid location is found
                        fallback_bbox_route = [
                            [bbox_sw_lon, bbox_sw_lat],
                            [bbox_ne_lon, bbox_ne_lat]
                        ]
                        route = clnt.directions(fallback_bbox_route, profile='driving-car', format='geojson', continue_straight=False)
                        route_coordinates = route['features'][0]['geometry']['coordinates']

            else:
                route_coordinates = []

        except ors_exceptions.ApiError as e:
            # Check for specific error code indicating no routable point found
            if 'could not find routable point' in str(e):
                print(f"\033[41mAPI Error for tweet {tweet_counter}: {e}\033[0m")
                skipped_tweets += 1
                return None, None
            else:
                print(f"\033[41mUnexpected API Error: {e}\033[0m")
                skipped_tweets += 1
                return None, None
        except Exception as e:
            print(f"\033[41mError processing tweet {tweet_counter}: {e}\033[0m")
            skipped_tweets += 1
            return None, None



    # Filter out None values to avoid 'NoneType' subscriptable error
    route_coordinates = [coord for coord in route_coordinates if coord is not None and len(coord) > 1]

    # Convert route coordinates to (latitude, longitude) format
    if(route_coordinates[0][0]>50):
        lat_lng_coordinates = [(coord[1], coord[0]) for coord in route_coordinates]

    else:
        lat_lng_coordinates = route_coordinates
    print("Route Coordinates:", lat_lng_coordinates)

    df.at[index, 'route_coordinates'] = str(route_coordinates)

    # df.to_csv("tweets_with_route_coordinates.csv", index=False)
    df.to_csv(REPL_PATH+"/Data/routes_data.csv", index=False)

    return lat_lng_coordinates


for index, row in df.iloc[0:5144].iterrows():
    route_coordinates = process_tweet(
        row['translated_text'],
        row['bbox_ne_lat'],
        row['bbox_ne_lon'],
        row['bbox_sw_lat'],
        row['bbox_sw_lon']
    )


# Display number of skipped tweets
# ANSI escape code for red background and white text
print(f"\033[41m\033[97mSkipped {skipped_tweets} tweets that could not be geocoded.\033[0m")

print("Route coordinates extracted and saved to 'routes_data.csv'")


<ipython-input-26-36d0affcca11>:15: DtypeWarning: Columns (6,7,37,38,39,40,41,42,43,44,45,46,47,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(REPL_PATH+"/Data/tweets_data.csv")


Skipped 0 tweets till now.
Processing Tweet 0: Traffic is now normal at Tilak Nagar towards Hari Nagar.
Tokenized Words: ['Traffic', 'is', 'now', 'normal', 'at', 'Tilak', 'Nagar', 'towards', 'Hari', 'Nagar']
POS Tags (NLTK): [('Traffic', 'NN'), ('is', 'VBZ'), ('now', 'RB'), ('normal', 'JJ'), ('at', 'IN'), ('Tilak', 'NNP'), ('Nagar', 'NNP'), ('towards', 'NNS'), ('Hari', 'NNP'), ('Nagar', 'NNP'), ('.', '.')]
Tilak Nagar LOC
Hari Nagar LOC
Locations: ['Tilak Nagar', 'Hari Nagar']
Extracted Prepositions: ['at', 'towards']
Directional Preposition: towards
config_num:  12
Locations: ['Tilak Nagar', 'Hari Nagar']
point: Tilak Nagar
direction: Hari Nagar
Route Coordinates: [(28.636712, 77.096411), (28.636558, 77.096024), (28.636273, 77.095207), (28.635757, 77.093773), (28.635703, 77.093623), (28.635512, 77.093209), (28.634897, 77.092), (28.634965, 77.091933), (28.635331, 77.092592), (28.635621, 77.093114), (28.635695, 77.093281), (28.635796, 77.093509), (28.636376, 77.095156), (28.63682, 77.09

<ipython-input-26-36d0affcca11>:221: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[[77.096411, 28.636712], [77.096024, 28.636558], [77.095207, 28.636273], [77.093773, 28.635757], [77.093623, 28.635703], [77.093209, 28.635512], [77.092, 28.634897], [77.091933, 28.634965], [77.092592, 28.635331], [77.093114, 28.635621], [77.093281, 28.635695], [77.093509, 28.635796], [77.095156, 28.636376], [77.096385, 28.63682], [77.099168, 28.638133], [77.099335, 28.638225], [77.09945, 28.638288], [77.099705, 28.638429], [77.101251, 28.639076], [77.102054, 28.639384], [77.102852, 28.63969], [77.10336, 28.639883], [77.104147, 28.640185], [77.105317, 28.64067], [77.105365, 28.64069], [77.105525, 28.640755], [77.1068, 28.641275], [77.107131, 28.641404], [77.107615, 28.64159], [77.107702, 28.641636], [77.107797, 28.641673], [77.107893, 28.641514], [77.107924, 28.641422], [77.107883, 28.641093], [77.107839, 28.641025], [77.1

Skipped 0 tweets till now.
Processing Tweet 1: Now Traffic is normal at AIIMS , in the carriageway running from AIIMS towards R.K. Puram .
Tokenized Words: ['Now', 'Traffic', 'is', 'normal', 'at', 'AIIMS', 'in', 'the', 'carriageway', 'running', 'from', 'AIIMS', 'towards', 'R.K.', 'Puram']
POS Tags (NLTK): [('Now', 'RB'), ('Traffic', 'NNP'), ('is', 'VBZ'), ('normal', 'JJ'), ('at', 'IN'), ('AIIMS', 'NNP'), (',', ','), ('in', 'IN'), ('the', 'DT'), ('carriageway', 'NN'), ('running', 'VBG'), ('from', 'IN'), ('AIIMS', 'NNP'), ('towards', 'NNS'), ('R.K.', 'NNP'), ('Puram', 'NNP'), ('.', '.')]
Traffic PERSON
AIIMS LOC
AIIMS LOC
R.K. Puram ORG
Locations: ['AIIMS', 'AIIMS', 'R.K. Puram']
Extracted Prepositions: ['at', 'in', 'from', 'towards']
Directional Preposition: from
Directional Preposition: towards
config_num:  12
Locations: ['AIIMS', 'AIIMS', 'R.K. Puram']
point: AIIMS
direction: R.K. Puram
IT'S ME WHO'S DOING IT NOW
True
Route Coordinates: [(28.545441, 77.180556), (28.545479, 77.180465),

<ipython-input-21-34b0bf298dae>:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  road_geometry = road_segments.unary_union  # Combines into a single MultiLineString


...

Route Coordinates: [(28.6418753, 77.1742995), (28.6419306, 77.1742565), (28.6419661, 77.1742214), (28.6420364, 77.1741447), (28.6420364, 77.1741447), (28.6422182, 77.1738953), (28.6425729, 77.1732841), (28.6593798, 77.1456271), (28.6593075, 77.145814), (28.6591468, 77.1461153), (28.6576722, 77.1481436), (28.6578042, 77.1477045), (28.6582076, 77.1470286), (28.6584107, 77.1466883), (28.6586117, 77.1463486), (28.6569371, 77.1495568), (28.6568706, 77.1496772), (28.6568597, 77.1496924), (28.6511512, 77.1588833), (28.651177, 77.1588391), (28.6596089, 77.1460517), (28.659523, 77.1459815), (28.6594636, 77.1459585), (28.6594071, 77.1459518), (28.6593588, 77.1459559), (28.6593047, 77.1459746), (28.6592542, 77.1460039), (28.6592023, 77.1460484), (28.6591468, 77.1461153), (28.6577292, 77.1482389), (28.6571706, 77.1491666), (28.6569852, 77.1494746), (28.6569767, 77.1494886), (28.6568925, 77.1493989), (28.657655, 77.1481714), (28.6576722, 77.1481436), (28.6591468, 77.1461153), (28.659032, 77.1462837

<ipython-input-21-34b0bf298dae>:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  road_geometry = road_segments.unary_union  # Combines into a single MultiLineString


Route Coordinates: [(28.6190168, 77.2094715), (28.6192383, 77.2096351), (28.61957, 77.2098801), (28.6200822, 77.2102472), (28.6302174, 77.2181923), (28.6302198, 77.2181247), (28.6302159, 77.218057), (28.6302022, 77.2179911), (28.6301792, 77.2179285), (28.6301476, 77.2178711), (28.6301081, 77.2178204), (28.6300619, 77.2177778), (28.6300101, 77.2177443), (28.6293363, 77.2172711), (28.6237991, 77.2129097), (28.6238697, 77.2130542), (28.6239194, 77.2131117), (28.6239756, 77.2131561), (28.6241953, 77.2133154), (28.6243036, 77.2133939), (28.6243867, 77.2134541), (28.6247825, 77.213741), (28.6248795, 77.2138113), (28.6251135, 77.2139809), (28.6254366, 77.2142151), (28.6255219, 77.2142769), (28.6260718, 77.2146413), (28.6265184, 77.2149372), (28.6265389, 77.2149546), (28.6266867, 77.2150807), (28.627, 77.2154323), (28.6269171, 77.2153759), (28.6202874, 77.2103967), (28.6204928, 77.2105484), (28.6230299, 77.2126312), (28.6230197, 77.212561), (28.6229988, 77.2124939), (28.622968, 77.2124319), (2

<ipython-input-21-34b0bf298dae>:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  road_geometry = road_segments.unary_union  # Combines into a single MultiLineString


Route Coordinates: [(28.6190168, 77.2094715), (28.6192383, 77.2096351), (28.61957, 77.2098801), (28.6200822, 77.2102472), (28.6302174, 77.2181923), (28.6302198, 77.2181247), (28.6302159, 77.218057), (28.6302022, 77.2179911), (28.6301792, 77.2179285), (28.6301476, 77.2178711), (28.6301081, 77.2178204), (28.6300619, 77.2177778), (28.6300101, 77.2177443), (28.6293363, 77.2172711), (28.6237991, 77.2129097), (28.6238697, 77.2130542), (28.6239194, 77.2131117), (28.6239756, 77.2131561), (28.6241953, 77.2133154), (28.6243036, 77.2133939), (28.6243867, 77.2134541), (28.6247825, 77.213741), (28.6248795, 77.2138113), (28.6251135, 77.2139809), (28.6254366, 77.2142151), (28.6255219, 77.2142769), (28.6260718, 77.2146413), (28.6265184, 77.2149372), (28.6265389, 77.2149546), (28.6266867, 77.2150807), (28.627, 77.2154323), (28.6269171, 77.2153759), (28.6202874, 77.2103967), (28.6204928, 77.2105484), (28.6230299, 77.2126312), (28.6230197, 77.212561), (28.6229988, 77.2124939), (28.622968, 77.2124319), (2